# tpu-retrieval-bench on Colab TPU

Runtime > Change runtime type > TPU v5e-1. One cell runs the whole Run-2 grid (20 repetitions per record) into a fresh `results/colab_run`, validates and summarizes it. Everything is zipped at the end.

In [ ]:
import sys
print(sys.version)
if sys.version_info < (3, 12):
    print('WARNING: python < 3.12. Fallback: pip install "jax[tpu]<0.11" and '
          'pip install --ignore-requires-python -e ".[data,plot]" (repo declares >= 3.12).')

In [ ]:
!git clone https://github.com/maxpetrusenko/tpu-retrieval-bench
%cd tpu-retrieval-bench
!pip install -q -e ".[data,plot]" "jax[tpu]==0.12.0.1" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
# jax pinned to the uv.lock version (0.12.0.1). If pip changed jax, restart the runtime
# (Runtime > Restart session) before running the next cell.

In [ ]:
# Run in a subprocess: importing jax in the notebook kernel would lock the TPU and make every
# later `!python` cell fail with "TPU is already in use".
!python -c "import jax; d = jax.devices(); assert d[0].platform == 'tpu', f'not a TPU runtime: {d}'; print(d[0].device_kind, len(d), 'device(s)')"

## Full Run-2 grid (single entry point)

Main grid, SIFT-1M, block sweeps, recall sweep and device time. Takes a while; `scripts/reproduce_all.sh` is also the local command.

In [ ]:
!bash scripts/reproduce_all.sh results/colab_run

## Profiler trace for one config

In [ ]:
# Script, not an in-kernel cell: the kernel must never import jax (it would hold the TPU).
!python bench/profile_one.py --m 128 --n 1000000 --d 128 --k 10 --out results/profile_jit_topk_m128_n1e6_d128

## Plots and download

In [ ]:
!python bench/plot.py

In [ ]:
!zip -qr results.zip results docs/img
from google.colab import files
files.download("results.zip")